# WO06 -- core-share exploration for the Lovejoy-region coherence badge

Notebook-only. No engine, API, or UI code changes come out of this notebook directly --
see `docs/design/_workbench/coherence_study_opus.md` (full spec) and
`docs/edop/workbench/workplan_africa.md` (WO06 summary). The shipped dispersion badge
(`coherence_iqr`, `59bfcea`) stays exactly as-is until a separate, later WO retires it.

**Why.** The shipped badge tests weighted IQR (p75-p25) < 20 on a region's per-basin
percentile scores. IQR counts inward from both ends, so a lopsided (skewed) distribution's
upper edge can land inside a thin tail and inflate the width even when the bulk of the mass
is tight. Confirmed live before this notebook was started: `snd_pc_sav`/`snd_pc_uav`
(visually near-identical, bulk-left/thin-tail-right) receive opposite badges for real
regions (Rivers: concentrated 12.74 / spread 20.88; North Coast: concentrated 15.79 /
spread 29.05).

**Proposed alternative -- core share.** The fraction of a region's weighted area falling
inside the densest window of fixed half-width delta on the global percentile (rank) axis.
Tail-immune by construction (material outside the window subtracts equally regardless of
how far out it lies).

**Naming note.** Core share as defined is a fixed-width relative of the shorth /
highest-density-interval family (shortest interval containing a fixed *mass*; core share
fixes the *width* and reports the mass instead). Worth confirming the write-up below uses
that established vocabulary rather than re-deriving it under a new name.

**Structure**: Part A (A1 compute, A4 tie/NaN validity checks -- run before interpreting
anything else, A2 stability, A3 visual validation against IQR), then Part B (exploratory
region-level profile, no threshold adopted).


In [ ]:
# Cell 1
%matplotlib inline
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import shapely.geometry
from scipy.stats import spearmanr
from IPython.display import Image as IPImage, display

from scripts.shared import db_utils
from scripts.shared.db_utils import db_connect
from scripts.edop.areas.engine import (
    load_catalog,
    resolve_polygon,
    attach_values,
    aggregate_b1,
    weighted_quantile,
    _BLOCK1_CLUSTERS,
    _LEVEL_TABLE,
    _LEVEL_VIEW,
)

ROOT = Path(db_utils.__file__).parent.parent.parent
OUT  = ROOT / 'output' / 'edop' / 'workbench'
OUT.mkdir(parents=True, exist_ok=True)

LEVEL = 6
DELTAS = [10, 15, 20]

print(f"OUT = {OUT}")


In [ ]:
# Cell 2
# Geometry: read the already-built, served lovejoy_regions.geojson directly -- one source
# of truth (same file the /api/lovejoy-signature route reads), no whg_staging touch here.
conn = db_connect()

geojson_path = ROOT / 'app' / 'static' / 'workbench' / 'lovejoy_regions.geojson'
lovejoy = json.loads(geojson_path.read_text())

regions = {}   # src_id -> {name, macro, geom_wkt}
for feat in lovejoy['features']:
    p = feat['properties']
    regions[p['src_id']] = {
        'name':     p['name'],
        'macro':    p['macro'],
        'geom_wkt': shapely.geometry.shape(feat['geometry']).wkt,
    }

print(f"{len(regions)} Lovejoy regions loaded from {geojson_path.name}")


In [ ]:
# Cell 3
warnings.filterwarnings("ignore", message="pandas only supports SQLAlchemy")

# Per-region: resolve the L6 basin set, attach percentile-score + raw-value matrices, and run
# aggregate_b1 (the exact shipped function -- same iqr/p10/p90/coherence the badge uses, no
# hand re-derivation). Cached in region_data so every later cell reuses this without re-hitting
# the DB. Expect ~1.5-4s/region (measured in WO05); ~2 min total for 34 regions.
meta_df = load_catalog(level=LEVEL)
table   = _LEVEL_TABLE[LEVEL]
view    = _LEVEL_VIEW[LEVEL]

region_data = {}   # src_id -> {basin_set, matrix_df, raw_df, b1_rows (dict by variable), n_units}
skipped = []       # src_id with zero L6 basin coverage (small offshore island groups, per WO05)

for src_id, r in regions.items():
    basin_set = resolve_polygon(r['geom_wkt'], f'{LEVEL:02d}', conn)
    if basin_set.empty:
        skipped.append(src_id)
        continue
    matrix_df, class_id_df, raw_df = attach_values(basin_set, meta_df, conn, table, view)
    b1_rows = {row['variable']: row for row in aggregate_b1(basin_set, matrix_df, meta_df)}
    region_data[src_id] = {
        'basin_set': basin_set,
        'matrix_df': matrix_df,
        'raw_df':    raw_df,
        'b1_rows':   b1_rows,
        'n_units':   len(basin_set),
    }
    print(f"{src_id:8s} {r['name']:28s} {len(basin_set):4d} basins")

print()
print(f"{len(region_data)} regions resolved, {len(skipped)} skipped (no L6 coverage): {skipped}")


## Part A -- does core share track visual reading?

### A1. Compute

Scope: the 34 B1 continuous variables (`kind == 'continuous'` and
`typology_cluster` in `{continental-gradient, scale-dependent}`) -- the exact variable set
`aggregate_b1` scores and badges today, which is what this study is about. (B5's separate,
short extreme-variable list -- e.g. `river_area` -- is out of scope here; a natural follow-on
if this generalizes.)

**Data source note.** `/api/lovejoy-signature`'s `detail.distribution` histogram
(`engine.py :: _weighted_histogram`) bins over each region's own local `[min, max]` score
range at a fixed 20 bins -- not the global `[0, 100]` rank axis a 1-rank sliding window
needs. So this notebook works from the raw per-basin weighted scores (`matrix_df` joined to
`basin_set`'s weights, masked + normalized -- the same three lines `aggregate_b1` runs
internally), not the API payload.


In [ ]:
# Cell 4
b1_meta = meta_df[(meta_df['kind'] == 'continuous') & (meta_df['typology_cluster'].isin(_BLOCK1_CLUSTERS))]
B1_VARS = list(b1_meta.index)
print(f"{len(B1_VARS)} B1 continuous variables in scope")


def scores_weights(basin_set, matrix_df, api_key):
    """Per-basin (score, normalized weight) for one variable in one region -- same
    mask/join/normalize `aggregate_b1` runs internally, exposed here for reuse across
    variables without recomputing badge rows.

    Returns (scores, wts_norm) as float arrays, or (None, None) if no valid coverage.
    """
    bs = basin_set.set_index('hybas_id') if 'hybas_id' in basin_set.columns else basin_set.copy()
    bs = bs.copy()
    bs.index = bs.index.astype('int64')
    md = matrix_df.copy()
    md.index = md.index.astype('int64')

    joined = bs[['weight']].join(md, how='inner')
    col = pd.to_numeric(joined[api_key], errors='coerce')
    w = joined['weight']
    mask = col.notna()

    scores = col[mask].values.astype(float)
    wts = w[mask].values.astype(float)
    coverage = wts.sum()
    if len(scores) == 0 or coverage == 0:
        return None, None
    return scores, wts / coverage


def core_share(scores, wts_norm, delta, step=1):
    """Max weighted share inside any window of width 2*delta on [0, 100], slid at
    `step`-rank increments (no mode estimation, no tie-breaking -- exhaustive scan).

    Returns (best_share, window_lo, window_hi). window_lo is None if scores is empty.
    """
    if scores is None or len(scores) == 0:
        return 0.0, None, None
    positions = np.arange(0, 100 - 2 * delta + step, step)
    best_share, best_lo = 0.0, None
    for lo in positions:
        hi = lo + 2 * delta
        share = wts_norm[(scores >= lo) & (scores <= hi)].sum()
        if share > best_share:
            best_share, best_lo = share, lo
    return float(best_share), (float(best_lo) if best_lo is not None else None), \
        (float(best_lo) + 2 * delta if best_lo is not None else None)


In [ ]:
# Cell 5
records = []
for src_id, data in region_data.items():
    basin_set, matrix_df, b1_rows = data['basin_set'], data['matrix_df'], data['b1_rows']
    for api_key in B1_VARS:
        row = b1_rows.get(api_key)
        if row is None or row.get('coherence') is None:
            continue   # no_data or outside_active_domain -- nothing to compare
        scores, wts_norm = scores_weights(basin_set, matrix_df, api_key)
        if scores is None:
            continue
        det = row['detail']
        for delta in DELTAS:
            share, lo, hi = core_share(scores, wts_norm, delta)
            records.append({
                'src_id': src_id, 'region': regions[src_id]['name'], 'variable': api_key,
                'band': row['band'], 'delta': delta,
                'core_share': share, 'window_lo': lo, 'window_hi': hi,
                'iqr': det['iqr'], 'p10': det['p10'], 'p90': det['p90'],
                'coherence_iqr': row['coherence'],
                'n_basins': row['n_units'], 'total_weight': data['n_units'],
            })

core_share_df = pd.DataFrame.from_records(records)
out_path = OUT / 'wo06_core_share.tsv'
core_share_df.to_csv(out_path, sep='\t', index=False)
print(f"{len(core_share_df)} rows ({core_share_df['src_id'].nunique()} regions x "
      f"{core_share_df['variable'].nunique()} variables x {len(DELTAS)} deltas) -> {out_path}")
print(core_share_df.head(9).to_string())


### A4. Ties and NoData -- check before interpreting anything

Two possible bugs that would corrupt every number above. **Read the printed results before
running A2/A3** -- if either check turns up a real problem, stop and report it rather than
proceeding to rank or plot on a corrupted axis.

1. Does the global percentile transform (`PERCENT_RANK()` in `engine.py :: rank_expr`) spread
   tied raw values across an interval, or give them one shared score? (Reading the SQL: SQL's
   `PERCENT_RANK` uses `RANK()`-style tie handling -- equal values get the identical rank --
   so it should *not* spread. Confirmed empirically below, not just from reading the code.)
2. Are NoData basins (`-9999`/`NULL`) excluded from both the weight sum and the window count,
   rather than contributing at any rank?


In [ ]:
# Cell 6 -- A4, tie check
# Pick the highest zero_fraction B1 variables (biggest expected tie blocks) and, across every
# cached region, group basins by raw value and check the score's spread *within* each raw-value
# group. If PERCENT_RANK shares one score per tie (expected), within-group score range == 0
# everywhere. raw_df and matrix_df are both indexed by api_key + hybas_id, so no db_col lookup
# needed.
tie_check_vars = b1_meta.sort_values('zero_fraction', ascending=False).head(4).index.tolist()
print(f"Tie-check variables (highest zero_fraction): {tie_check_vars}")
print()

worst_spread = 0.0
worst_ctx = None
largest_tie_block = 0
for api_key in tie_check_vars:
    for src_id, data in region_data.items():
        raw_df, matrix_df = data['raw_df'], data['matrix_df']
        if api_key not in raw_df.columns or api_key not in matrix_df.columns:
            continue
        both = pd.DataFrame({'raw': raw_df[api_key], 'score': matrix_df[api_key]}).dropna()
        if both.empty:
            continue
        grouped = both.groupby('raw')['score'].agg(['nunique', 'count', lambda s: s.max() - s.min()])
        grouped.columns = ['n_unique_scores', 'n_basins', 'score_range']
        tie_groups = grouped[grouped['n_basins'] > 1]
        if tie_groups.empty:
            continue
        largest_tie_block = max(largest_tie_block, int(tie_groups['n_basins'].max()))
        max_spread_here = float(tie_groups['score_range'].max())
        if max_spread_here > worst_spread:
            worst_spread = max_spread_here
            worst_ctx = (api_key, src_id, int(tie_groups['n_basins'].max()))

print(f"Largest tie block found (any variable/region): {largest_tie_block} basins sharing one raw value")
print(f"Worst within-tie-block score range found: {worst_spread:.6f}"
      f"{'' if worst_spread == 0 else f'  <-- NOT ZERO, see {worst_ctx}'}")
print()
print("PASS: ties share one score, not spread." if worst_spread == 0
      else "FAIL: tie spreading detected -- stop, do not run A2/A3 until this is resolved.")


In [ ]:
# Cell 7 -- A4, NaN/NoData check
# Confirm raw NoData (-9999 or NULL) basins never carry a score, and never survive into
# scores_weights()'s masked output. attach_values() should already convert -9999 -> NaN in
# raw_df (view-level), but check the sentinel too in case a variable bypasses that path.
mismatches = []
for api_key in B1_VARS:
    for src_id, data in region_data.items():
        raw_df, matrix_df = data['raw_df'], data['matrix_df']
        if api_key not in raw_df.columns or api_key not in matrix_df.columns:
            continue
        raw = pd.to_numeric(raw_df[api_key], errors='coerce')
        score = matrix_df[api_key]
        raw_is_nodata = raw.isna() | (raw == -9999)
        score_is_nan = score.isna()
        if not (raw_is_nodata == score_is_nan).all():
            n_bad = int((raw_is_nodata != score_is_nan).sum())
            mismatches.append((api_key, src_id, n_bad))

print(f"NoData/score mismatches found: {len(mismatches)}")
if mismatches:
    print(mismatches[:10])

# Spot-check scores_weights() itself never returns a NoData basin for one B1 var/region.
sample_src, sample_data = next(iter(region_data.items()))
sample_var = B1_VARS[0]
raw = pd.to_numeric(sample_data['raw_df'][sample_var], errors='coerce')
n_nodata_raw = int((raw.isna() | (raw == -9999)).sum())
scores, wts_norm = scores_weights(sample_data['basin_set'], sample_data['matrix_df'], sample_var)
n_returned = len(scores) if scores is not None else 0
print()
print(f"Spot check ({sample_var}, {sample_src}): {n_nodata_raw} NoData raw basins, "
      f"{n_returned} returned by scores_weights() out of {sample_data['n_units']} total "
      f"({sample_data['n_units'] - n_nodata_raw} expected)")
print()
print("PASS: NoData excluded from both score and weight." if not mismatches
      else "FAIL: NoData leakage detected -- stop, do not run A2/A3 until this is resolved.")


### A2. Stability across delta

Only proceed past this point if both A4 checks above printed PASS.

Rank the variables by core share within each region, separately at each delta, and check
the correlation between those orderings. Stable ordering across delta means delta is a
legibility choice, not a source of disagreement; churn means the distributions are too
irregular for a one-number summary and that is a valid, reportable negative outcome.


In [ ]:
# Cell 8
# Per region: Spearman correlation between the variable ranking at delta=10 vs 15, and 15 vs 20.
delta_pairs = [(10, 15), (15, 20), (10, 20)]
stability_records = []
for src_id, g in core_share_df.groupby('src_id'):
    piv = g.pivot(index='variable', columns='delta', values='core_share')
    for d1, d2 in delta_pairs:
        if d1 not in piv.columns or d2 not in piv.columns:
            continue
        pair = piv[[d1, d2]].dropna()
        if len(pair) < 3:
            continue
        rho, _ = spearmanr(pair[d1], pair[d2])
        stability_records.append({'src_id': src_id, 'region': regions[src_id]['name'],
                                   'delta_pair': f'{d1}-{d2}', 'spearman_rho': rho, 'n_vars': len(pair)})

stability_df = pd.DataFrame.from_records(stability_records)
print(stability_df.groupby('delta_pair')['spearman_rho'].describe().to_string())
print()
low = stability_df[stability_df['spearman_rho'] < 0.7].sort_values('spearman_rho')
print(f"{len(low)} region/delta-pair combos with rho < 0.7 (ordering churns):")
print(low.to_string() if len(low) else "(none)")


### A3. Where the two statistics disagree

Pick one delta for the rest of Part A (the one A2 showed as most stable / most legible;
default to 15 -- the middle value -- unless A2 says otherwise). Scatter core share against
IQR, then hand-inspect the strongest disagreements against the histogram as it actually
renders in the UI. **Acceptance gate: Karl's own eye against this panel, not another
metric.**


In [ ]:
# Cell 9
DELTA_MAIN = 15   # revisit after reading A2's output above

print("drawing core_share vs iqr scatter...")
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.patch.set_facecolor('white')
for ax, delta in zip(axes, DELTAS):
    d = core_share_df[core_share_df['delta'] == delta]
    ax.set_facecolor('white')
    colors = d['coherence_iqr'].map({'concentrated': '#2c7a3d', 'spread': '#b5651d'})
    ax.scatter(d['iqr'], d['core_share'], s=10, alpha=0.5, c=colors)
    ax.set_title(f"delta={delta}", color='black')
    ax.set_xlabel('IQR (shipped)', color='black')
    ax.set_ylabel('core share', color='black')
    ax.tick_params(colors='black')
outpath = OUT / 'wo06_a3_scatter.png'
fig.savefig(outpath, facecolor='white', dpi=120, bbox_inches='tight')
plt.close(fig)
display(IPImage(str(outpath)))


In [ ]:
# Cell 10
# Disagreement = gap between the two statistics' percentile-rank "tightness" within this
# dataset (core_share: higher = tighter; iqr: lower = tighter, so its rank is inverted before
# comparing). The interesting off-diagonal case this study exists to catch: tight core share
# (high pct) with wide IQR (also high pct after inversion) -- skew.
d = core_share_df[core_share_df['delta'] == DELTA_MAIN].copy()
d['core_share_pct'] = d['core_share'].rank(pct=True)
d['iqr_tightness_pct'] = (-d['iqr']).rank(pct=True)   # invert: low iqr -> high pct
d['disagreement'] = (d['core_share_pct'] - d['iqr_tightness_pct']).abs()

top20 = d.sort_values('disagreement', ascending=False).head(20)

# Force in the motivating pair for every region where it appears, regardless of rank.
motivating = d[d['variable'].isin(['pct_sand', 'pct_sand_upstream'])]
inspection_set = pd.concat([top20, motivating]).drop_duplicates(subset=['src_id', 'variable'])

print(f"Top 20 disagreements (delta={DELTA_MAIN}) + motivating pct_sand/pct_sand_upstream "
      f"rows ({len(inspection_set)} total to inspect):")
print(inspection_set[['region', 'variable', 'core_share', 'iqr', 'coherence_iqr', 'disagreement']]
      .sort_values('disagreement', ascending=False).to_string())


In [ ]:
# Cell 11
# Annotated histogram per disagreement: weighted score distribution on the fixed [0,100]
# global rank axis (the axis core share and IQR both operate on -- more legible here than the
# UI's per-region-local-range histogram), with the winning core-share window shaded and both
# numbers + the shipped badge in the title.
rows_to_plot = inspection_set.sort_values('disagreement', ascending=False).to_dict('records')
n = len(rows_to_plot)
ncols = 5
nrows = -(-n // ncols)

print(f"drawing {n} annotated disagreement histograms...")
fig, axes = plt.subplots(nrows, ncols, figsize=(3.2 * ncols, 2.6 * nrows))
fig.patch.set_facecolor('white')
axes = np.atleast_1d(axes).flatten()

for ax, rec in zip(axes, rows_to_plot):
    ax.set_facecolor('white')
    scores, wts_norm = scores_weights(
        region_data[rec['src_id']]['basin_set'], region_data[rec['src_id']]['matrix_df'], rec['variable'])
    ax.hist(scores, bins=20, range=(0, 100), weights=wts_norm, color='#5b8dc4', alpha=0.8)
    if rec['window_lo'] is not None:
        ax.axvspan(rec['window_lo'], rec['window_hi'], color='#2c7a3d', alpha=0.15)
    ax.set_title(f"{rec['region'][:14]} / {rec['variable']}\n"
                 f"core={rec['core_share']:.0%} iqr={rec['iqr']:.1f} ({rec['coherence_iqr']})",
                 fontsize=8, color='black')
    ax.set_xlim(0, 100)
    ax.tick_params(labelsize=7, colors='black')

for ax in axes[n:]:
    ax.axis('off')

fig.tight_layout()
outpath = OUT / 'wo06_a3_disagreements.png'
fig.savefig(outpath, facecolor='white', dpi=120, bbox_inches='tight')
plt.close(fig)
display(IPImage(str(outpath)))


In [ ]:
# Cell 12
# Control panel: ~10 pairs where the two statistics agree, split tight/broad, so the visual
# comparison isn't drawn only from cases pre-selected for disagreement.
agree = d.sort_values('disagreement').head(200)   # most-agreeing pool to split from
agreed_tight = agree[agree['coherence_iqr'] == 'concentrated'].nsmallest(5, 'iqr')
agreed_broad = agree[agree['coherence_iqr'] == 'spread'].nlargest(5, 'iqr')
control_set = pd.concat([agreed_tight, agreed_broad]).to_dict('records')

print(f"drawing {len(control_set)} agreement-control histograms...")
fig, axes = plt.subplots(2, 5, figsize=(3.2 * 5, 2.6 * 2))
fig.patch.set_facecolor('white')
axes = axes.flatten()

for ax, rec in zip(axes, control_set):
    ax.set_facecolor('white')
    scores, wts_norm = scores_weights(
        region_data[rec['src_id']]['basin_set'], region_data[rec['src_id']]['matrix_df'], rec['variable'])
    ax.hist(scores, bins=20, range=(0, 100), weights=wts_norm, color='#5b8dc4', alpha=0.8)
    if rec['window_lo'] is not None:
        ax.axvspan(rec['window_lo'], rec['window_hi'], color='#2c7a3d', alpha=0.15)
    ax.set_title(f"{rec['region'][:14]} / {rec['variable']}\n"
                 f"core={rec['core_share']:.0%} iqr={rec['iqr']:.1f} ({rec['coherence_iqr']})",
                 fontsize=8, color='black')
    ax.set_xlim(0, 100)
    ax.tick_params(labelsize=7, colors='black')

for ax in axes[len(control_set):]:
    ax.axis('off')

fig.tight_layout()
outpath = OUT / 'wo06_a3_control.png'
fig.savefig(outpath, facecolor='white', dpi=120, bbox_inches='tight')
plt.close(fig)
display(IPImage(str(outpath)))


## Part B -- region-level profile (exploratory, no gate)

No threshold is being adopted here -- this exists to show whether a region-level
interpretive move ("on how many, and which, dimensions does a declared region actually
cohere environmentally?") has anything to say before any of it is built. Uses
`DELTA_MAIN` and a provisional core-share cutoff, both clearly labeled as such.


In [ ]:
# Cell 13
CUTOFF = 0.5   # provisional -- "coherent on this variable" if core_share >= CUTOFF at DELTA_MAIN

d_main = core_share_df[core_share_df['delta'] == DELTA_MAIN].copy()
d_main['coherent'] = d_main['core_share'] >= CUTOFF

per_region = d_main.groupby(['src_id', 'region']).agg(
    n_variables=('variable', 'count'),
    n_coherent=('coherent', 'sum'),
    n_basins=('n_basins', 'first'),
).reset_index()
per_region['pct_coherent'] = per_region['n_coherent'] / per_region['n_variables']
per_region = per_region.sort_values('n_coherent', ascending=False)

print(f"Regions ranked by count of variables clearing core_share >= {CUTOFF} "
      f"(delta={DELTA_MAIN}), most to least environmentally coherent:")
print(per_region.to_string(index=False))

# By band, per region
by_band = d_main.groupby(['src_id', 'region', 'band'])['coherent'].sum().unstack(fill_value=0)
print()
print("By band:")
print(by_band.to_string())


In [ ]:
# Cell 14
# Which variables do the coherence work -- if a handful of variables carry most regions'
# coherence, or if soil/terrain vars cohere while hydrology doesn't, that is a finding about
# the Lovejoy vocabulary, not about the statistic.
per_variable = d_main.groupby(['variable', 'band']).agg(
    n_regions=('coherent', 'count'),
    n_coherent_regions=('coherent', 'sum'),
    mean_core_share=('core_share', 'mean'),
).reset_index()
per_variable['pct_regions_coherent'] = per_variable['n_coherent_regions'] / per_variable['n_regions']
per_variable = per_variable.sort_values('pct_regions_coherent', ascending=False)

print("Variables ranked by share of regions where they clear the coherence cutoff:")
print(per_variable.to_string(index=False))
print()
print("By band, mean % of regions coherent:")
print(per_variable.groupby('band')['pct_regions_coherent'].mean().sort_values(ascending=False).to_string())


In [ ]:
# Cell 15
# Does coherence track region size? If strongly, any future cutoff has to control for it --
# the honest display may be a region's position relative to comparable-sized regions, not an
# absolute count.
rho_count, p_count = spearmanr(per_region['n_basins'], per_region['n_coherent'])
rho_pct, p_pct = spearmanr(per_region['n_basins'], per_region['pct_coherent'])
print(f"n_basins vs n_coherent (raw count):      rho={rho_count:.3f}  p={p_count:.4f}")
print(f"n_basins vs pct_coherent (normalized):    rho={rho_pct:.3f}  p={p_pct:.4f}")

print("drawing region size vs coherence...")
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
fig.patch.set_facecolor('white')
for ax, ycol, ylabel in zip(axes, ['n_coherent', 'pct_coherent'], ['# coherent variables', '% coherent variables']):
    ax.set_facecolor('white')
    ax.scatter(per_region['n_basins'], per_region[ycol], s=20, color='#2c5f8a')
    ax.set_xlabel('n_basins (region size)', color='black')
    ax.set_ylabel(ylabel, color='black')
    ax.tick_params(colors='black')
outpath = OUT / 'wo06_b_size_vs_coherence.png'
fig.savefig(outpath, facecolor='white', dpi=120, bbox_inches='tight')
plt.close(fig)
display(IPImage(str(outpath)))


In [ ]:
# Cell 16
# Regions x variables heatmap, shaded by core share -- the artifact most likely to be legible
# at Braga scale. Rows ordered by per_region's coherence ranking (Cell 13); columns by
# per_variable's coherence ranking (Cell 14), so structure (if any) reads on both axes.
region_order = per_region['region'].tolist()
var_order = per_variable['variable'].tolist()
heat = d_main.pivot(index='region', columns='variable', values='core_share').reindex(
    index=region_order, columns=var_order)

print("drawing regions x variables core-share heatmap...")
fig, ax = plt.subplots(figsize=(0.35 * len(var_order) + 3, 0.28 * len(region_order) + 2))
fig.patch.set_facecolor('white')
ax.set_facecolor('white')
im = ax.imshow(heat.values, aspect='auto', cmap='RdYlGn', vmin=0, vmax=1)
ax.set_xticks(range(len(var_order)))
ax.set_xticklabels(var_order, rotation=90, fontsize=6, color='black')
ax.set_yticks(range(len(region_order)))
ax.set_yticklabels(region_order, fontsize=6, color='black')
cbar = fig.colorbar(im, ax=ax, fraction=0.02, pad=0.01)
cbar.set_label('core share', color='black')
cbar.ax.yaxis.set_tick_params(color='black')
outpath = OUT / 'wo06_b_heatmap.png'
fig.savefig(outpath, facecolor='white', dpi=130, bbox_inches='tight')
plt.close(fig)
display(IPImage(str(outpath)))


## Wrap-up

**Part A gate**: review Cell 11 (disagreements) against Cell 12 (control) by eye. If core
share tracks your reading where IQR does not, Part A passes and a follow-on WO would propose
the display change (still not this notebook's job). If it doesn't, that's the deliverable --
report it as such.

**Deferred, untouched by this WO** (per the spec): reference-distribution question (raw vs.
rank), threshold derivation, marginal-basin trimming, `two_regime` interaction.

**Not done in this pass** -- flag if wanted: B5's short extreme-variable list (out of A1's
scope, see the note there); a formal write-up of the A2/A3 findings back into
`docs/design/_workbench/coherence_study_opus.md` or a `wo06_findings.md`.
